# WritingPrompts Story Generation

Generates LLM stories from the WritingPrompts dataset (Fan et al., 2018)

**Models**

| Key | Model |
|---|---|
| `llama` | meta-llama/Llama-3.1-8B-Instruct |
| `mistral` | mistralai/Mistral-7B-Instruct-v0.3 |
| `qwen` | Qwen/Qwen2.5-7B-Instruct |
| `gemma` | google/gemma-2-9b-it |

All four are open-weight models run locally via vLLM

**Setup***

- GPU: a L4 or A100 is recommended
- Run one model per session
- vLLM does not release GPU memory reliably, so after each model restart the runtime and change `MODEL_KEY`.

**Decoding configuration**

Fixed across all models: 
- `temperature=1.0`
- `top_p=0.9`
- `repetition_penalty=1.0` (disabled)
- `n=5`.

No `no_repeat_ngram` constraint is used (would directly manipulate lexical diversity)

## Installation

In [ ]:
# Restart the runtime after this cell
!pip install -q -U vllm datasets
!pip install -q --upgrade huggingface_hub
!pip uninstall -y -q torchaudio torchvision

In [ ]:
import json
import pathlib
import random
import re
import time
from google.colab import drive


# Mount Drive
drive.mount('/content/drive')
OUT_DIR = pathlib.Path('/content/drive/MyDrive/Masterarbeit/generations')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output directory:', OUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output directory: /content/drive/MyDrive/Masterarbeit/generations


## Configuration

In [ ]:
# Switch this between runs: 'llama' | 'mistral' | 'qwen' | 'gemma'
MODEL_KEY = 'mistral'


MODELS = {
    'llama':   'meta-llama/Llama-3.1-8B-Instruct',
    'mistral': 'mistralai/Mistral-7B-Instruct-v0.3',
    'qwen':    'Qwen/Qwen2.5-7B-Instruct',
    'gemma':   'google/gemma-2-9b-it',
}

MODEL_ID = MODELS[MODEL_KEY]

# Sampling
N_PROMPTS = 100   # number of prompts drawn from the test split
N_SAMPLES = 5     # generations per prompt (n=5)
SEED = 42         # controls both prompt selection and generation


# Decoding (identical for all four models)
TEMPERATURE = 1.0
TOP_P = 0.9
MAX_NEW_TOKENS = 1280     # Need to be high enough to allow for long stories
REPETITION_PENALTY = 1.0  # deliberately disabled


random.seed(SEED)

print(f'Model:  {MODEL_ID}')
print(f'Output: {N_PROMPTS} prompts x {N_SAMPLES} samples = {N_PROMPTS * N_SAMPLES} stories')

In [ ]:
# Llama and Gemma are gated (accept the license on each model page and store
# token in a .env file on Drive):
#   /content/drive/MyDrive/Masterarbeit/.env
# containing a single line:
#   HF_TOKEN=hf_xxxxxxxxxxxxxxxxxxxx

if MODEL_KEY in ('llama', 'gemma'):
    from huggingface_hub import login

    env_path = OUT_DIR.parent / '.env'

    # Read Hugging Face token from .env file on Drive
    hf_token = dict(
        line.strip().split('=', 1)
        for line in env_path.read_text().splitlines()
        if line.strip()
    )['HF_TOKEN']

    login(token=hf_token)
    print('Logged in to Hugging Face')

## Load WritingPrompts

- Original release (Fan et al., 2018) ships as `.wp_source` and `.wp_target` files
- Use HuggingFace to get dataset
- Stories in the original are pre-tokenized: `<newline>` markers, spaces before punctuation. 

In [ ]:
HF_REPO = 'euclaise/writingprompts'


def detokenize(text):
    """Undo the pre-tokenization of the raw WritingPrompts files."""

    # Replace <newline> with actual newlines
    text = text.replace('<newline>', '\n')

    # Remove spaces before punctuation
    text = re.sub(r'\s+([,.!?;:\'])', r'\1', text)

    # Collapse multiple newlines
    text = re.sub(r'\s*\n\s*', '\n', text)

    # Collapse multiple spaces and tabs into a single space
    text = re.sub(r'[ \t]{2,}', ' ', text)

    return text.strip()


def strip_tag(prompt):
    """Remove Reddit tags such as [WP], [EU], [W.P] from the start of a prompt.

    Loops because prompts sometimes carry stacked tags
    Allows periods because variants like [W.P] occur.
    """

    # Pattern matches tags like [WP], [EU], [W.P], [WP.], etc., with optional 
    # whitespace around the brackets and inside the brackets
    pattern = r'^\s*\[\s*[A-Za-z.]{2,4}\s*\]\s*'

    while re.match(pattern, prompt):
        prompt = re.sub(pattern, '', prompt)

    return prompt.strip()

In [ ]:
from datasets import load_dataset

# Load the WritingPrompts dataset from Hugging Face
ds = load_dataset(HF_REPO, split='test')
print('Columns:', ds.column_names)

# Column names vary by mirror: prompt/story or text/target
prompt_col = 'prompt' if 'prompt' in ds.column_names else ds.column_names[0]
story_col = 'story' if 'story' in ds.column_names else ds.column_names[1]

# Create a list of dictionaries containing prompt-story pairs
records = [{'prompt': r[prompt_col], 'story': r[story_col]} for r in ds]

print(f'Loaded {len(records)} prompt-story pairs from the test split')

Columns: ['prompt', 'story']
Loaded 15138 prompt-story pairs from the test split


In [ ]:
# Deduplicate prompts: test split contains ~2.8 stories per prompt (same prompt appears several times)

seen = set()
unique = []

for record in records:
    prompt = strip_tag(detokenize(record['prompt']))

    # Filter out empty prompts, duplicates
    if not prompt or prompt in seen:
        continue

    # Filter out prompts that are too short (<5 words) or too long (>80 words)
    n_words = len(prompt.split())
    if n_words < 5 or n_words > 80:
        continue

    # Add the unique prompt and its corresponding story to the list
    seen.add(prompt)
    unique.append({
        'prompt': prompt,
        'human_story': detokenize(record['story']),
    })


print(f'{len(unique)} unique prompts after filtering')

5232 unique prompts after filtering


In [ ]:
# Draw the sample once and reuse it for all four models, so every model uses same prompts

sampled = random.Random(SEED).sample(unique, N_PROMPTS)

for i, item in enumerate(sampled):
    item['prompt_id'] = f'wp_{i:04d}'


# Save the sampled prompts to a JSONL file for later use
with open(OUT_DIR / 'prompts_sample.jsonl', 'w') as f:
    for item in sampled:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')


print(f'Saved {len(sampled)} prompts')
print()
print('Example:')
print(sampled[0]['prompt'])

Saved 100 prompts

Example:
The first line of your story must include'and I caught him with, of all things, a fishing pole'.


## Prompt template

- Identical across models
- Without length instruction models produce very differently sized outputs

In [ ]:
SYSTEM_PROMPT = 'You are a creative writer contributing to an online short story forum.'

USER_TEMPLATE = (
    'Write a short story based on the following writing prompt.\n\n'
    'Prompt: {prompt}\n\n'
    'Write approximately 500 words. Output only the story, with no title, '
    'no preamble, and no commentary.'
)


def build_messages(prompt):
    user_content = USER_TEMPLATE.format(prompt=prompt)

    # Gemma's chat template has no 'system' role (only 'user'/'model'), so the
    # system prompt is folded into the user turn instead.
    
    if MODEL_KEY == 'gemma':
        return [
            {'role': 'user', 'content': f'{SYSTEM_PROMPT}\n\n{user_content}'},
        ]

    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_content},
    ]

## Generation (vLLM)

Passing `n=5` in `SamplingParams` and not five separate requests

In [ ]:
# Check GPU availability
!nvidia-smi

Tue Aug  4 15:11:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   46C    P0             26W /   72W |   21062MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Check installed versions of torch and vllm
!pip show torch vllm | grep -E "Name|Version"
!find / -name "libcudart.so*"

Name: torch
Version: 2.11.0+cu128
Name: vllm
Version: 0.26.0
/usr/local/lib/python3.12/dist-packages/nvidia/cuda_runtime/lib/libcudart.so.12
/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib/libcudart.so.13
/usr/local/cuda-12.8/targets/x86_64-linux/lib/libcudart.so.12
/usr/local/cuda-12.8/targets/x86_64-linux/lib/libcudart.so.12.8.90
/usr/local/cuda-12.8/targets/x86_64-linux/lib/libcudart.so
find: ‘/proc/91/task/91/net’: Invalid argument
find: ‘/proc/91/net’: Invalid argument
find: ‘/proc/60096’: No such file or directory


In [ ]:
# Load the CUDA runtime library with global symbol visibility so that vLLM can find it
import ctypes
ctypes.CDLL(
    '/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib/libcudart.so.13',
    mode=ctypes.RTLD_GLOBAL,
)

import sys

# Jupyter/Colab's stdout and stderr (ipykernel.iostream.OutStream) do not
# implement fileno(), but vLLM calls sys.stdout.fileno() to briefly silence
# NCCL's native logging when it sets up the process group. Fake the real
# OS-level fd numbers (1 = stdout, 2 = stderr) so that call succeeds.
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2

import glob, os

# Keep vLLM's engine in this process instead of a spawned subprocess. In a
# spawned child, ipykernel re-installs its own OutStream as sys.stdout, so
# the fileno() patch above (applied only in this process) would not carry
# over and the same crash would recur inside the child.
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

nvidia_lib_dirs = glob.glob("/usr/local/lib/python3.12/dist-packages/nvidia/*/lib")
os.environ["LD_LIBRARY_PATH"] = ":".join(nvidia_lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")])
print(nvidia_lib_dirs)


from vllm import LLM, SamplingParams
from transformers import AutoTokenizer


['/usr/local/lib/python3.12/dist-packages/nvidia/nvjitlink/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/cusolver/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/cuda_nvrtc/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/nvtx/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/cufft/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/cusparselt/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/nvshmem/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/cufile/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/cuda_cccl/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/cuda_nvcc/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/nccl/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/cuda_cupti/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/cusparse/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/curand/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/cublas/lib', '/u

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Apply each model's own chat template and hand plain text to vLLM
text_prompts = [
    tokenizer.apply_chat_template(
        build_messages(item['prompt']),
        tokenize=False,
        add_generation_prompt=True,
    )
    for item in sampled
]

print(f'Prepared {len(text_prompts)} prompts')

Prepared 100 prompts


In [ ]:
# Install torchvision for vLLM's image generation support (even if not used here)
!pip install -q torchvision --index-url https://download.pytorch.org/whl/cu128 --no-deps


In [ ]:
# Clear vLLM's model registry cache to ensure that the latest model classes are used
from vllm.model_executor.models import registry
registry._try_inspect_model_cls.cache_clear()

In [ ]:
# Initialize the vLLM model: use bfloat16 for speed and memory efficiency, and limit GPU memory utilization to 90%
llm = LLM(
    model=MODEL_ID,
    dtype='bfloat16',
    gpu_memory_utilization=0.90,
    max_model_len=2048,
    seed=SEED,
)

# Set up the sampling parameters for text generation
sampling_params = SamplingParams(
    n=N_SAMPLES,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_NEW_TOKENS,
    repetition_penalty=REPETITION_PENALTY,
    seed=SEED,
)

# Generate stories for the sampled prompts using vLLM
start = time.time()
outputs = llm.generate(text_prompts, sampling_params)

print(f'Generation took {time.time() - start:.1f}s')

INFO 08-04 15:12:42 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 42, 'max_model_len': 2048, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'mistralai/Mistral-7B-Instruct-v0.3'}
WARNING 08-04 15:12:43 [arg_utils.py:1631] The global random seed is set to 42. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 08-04 15:12:45 [config.py:385] Inferred from consolidated*.safetensors files torch.bfloat16 dtype.
INFO 08-04 15:12:46 [model.py:623] Resolved architecture: MistralForCausalLM
INFO 08-04 15:12:46 [model.py:1788] Using max model len 2048
INFO 08-04 15:12:46 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 08-04 15:12:46 [vllm.py:1109] Asynchronous scheduling is enabled.
INFO 08-04 15:12:46 [kernel.py:295] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native']

INFO 08-04 15:12:49 [core.py:116] Initializing a V1 LLM engine (v0.26.0) with config: model='mistralai/Mistral-7B-Instruct-v0.3', speculative_config=None, tokenizer='mistralai/Mistral-7B-Instruct-v0.3', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_en

ValueError: Free memory on device cuda:0 (1.28/22.03 GiB) on startup is less than desired GPU memory utilization (0.9, 19.83 GiB). Decrease GPU memory utilization or reduce GPU memory used by other processes.

In [ ]:
results = []

# Collect the generated stories and their metadata into a list of dictionaries
for item, output in zip(sampled, outputs):
    for k, candidate in enumerate(output.outputs):
        results.append({
            'prompt_id': item['prompt_id'],
            'prompt': item['prompt'],
            'model': MODEL_ID,
            'sample_idx': k,
            'story': candidate.text.strip(),
            'finish_reason': candidate.finish_reason,
            'n_tokens': len(candidate.token_ids),
        })

print(f'Collected {len(results)} stories')

In [ ]:
# Language drift check: flag stories that contain non-Latin script (CJK = Chinese, Japanese, Korean)

CJK = re.compile(r'[\u4e00-\u9fff\u3040-\u30ff\uac00-\ud7af]')

for record in results:
    hits = CJK.findall(record['story'])
    record['cjk_chars'] = len(hits)
    record['language_ok'] = len(hits) == 0

flagged = [r for r in results if not r['language_ok']]

print(f'Stories with non-Latin script: {len(flagged)} of {len(results)}')

for record in flagged:
    share = record['cjk_chars'] / len(record['story'])
    print(f"  {record['prompt_id']}/{record['sample_idx']}  {share:.0%} CJK")

## Save results

In [ ]:
out_path = OUT_DIR / f'writingprompts_{MODEL_KEY}_n{N_SAMPLES}.jsonl'

# Save the generated stories to a JSONL file
with open(out_path, 'w') as f:
    for record in results:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')


# Store the configuration
config = {
    'model_id': MODEL_ID,
    'dataset': 'WritingPrompts (Fan et al., 2018), test split',
    'n_prompts': N_PROMPTS,
    'n_samples': N_SAMPLES,
    'seed': SEED,
    'temperature': TEMPERATURE,
    'top_p': TOP_P,
    'max_new_tokens': MAX_NEW_TOKENS,
    'repetition_penalty': REPETITION_PENALTY,
    'system_prompt': SYSTEM_PROMPT,
    'user_template': USER_TEMPLATE,
}

with open(OUT_DIR / f'config_{MODEL_KEY}.json', 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)


print('Saved:', out_path)

## Sanity checks

- `finish_reason == 'length'`: story was cut off mid-sentence    
-> raise `MAX_NEW_TOKENS` or trim to the last complete sentence.
- Length distribution per model: systematic differences
- Preambles: like "Here is a story:" appear despite the instruction

In [ ]:
import collections
import statistics as st

# Compute and print summary statistics for the generated stories
lengths = [len(record['story'].split()) for record in results]
truncated = sum(1 for record in results if record['finish_reason'] == 'length')

print(f'Stories:      {len(results)}')
print(f'Median words: {st.median(lengths):.0f}')
print(f'Mean words:   {st.mean(lengths):.0f} (SD {st.stdev(lengths):.0f})')
print(f'Min / max:    {min(lengths)} / {max(lengths)}')
print(f'Truncated:    {truncated} ({100 * truncated / len(results):.1f}%)')

In [ ]:
# Look for repeated openings

openings = collections.Counter(' '.join(r['story'].split()[:3]) for r in results)

print('Most frequent openings:')
for opening, count in openings.most_common(5):
    print(f'  {count:3d}x  {opening}')

print()
print('--- Sample story ---')
print(results[0]['story'][:600])